# 08. Ocena niepewności TTA per wycinek

**Pytanie:** czy miary niepewności z augmentacji D4 (notebook 07) lepiej wskazują trudne miejsca, gdy liczymy je
na **wycinkach** obrazu, a nie na całym obrazie? Obraz MoNuSeg 1000×1000 ma kilkaset jąder, więc średnia po
całym obrazie rozmywa lokalne, trudne fragmenty.

**Jak:**
1. TTA liczymy na **całym obrazie** (pełny kontekst, bez sztucznych brzegów); predykcje bierzemy z cache notebooka 07.
2. Obraz dzielimy na siatkę wycinków `TILE × TILE` (domyślnie 250 px, czyli 4×4 = 16 wycinków).
3. Jądro należy do wycinka, w którym leży jego **środek** (jądro przecięte granicą liczy się raz, nie jest przycinane).
4. Wynik wycinka (tylko ze studenta, może go używać selektor):
   - `K_mean`, `U_MI_mean`, `Top_mean`: średnie miar jąder w wycinku,
   - `unstable = frac_D_lt1 + E_extra`: odsetek jąder z $D<1$ plus lokalne jądra dodatkowe
     (miara selektora `tta_kd`, tylko liczona na wycinku),
   - `rank_kd`: średnia rang procentowych `K_mean` i `unstable` (jak w `tta_kd`).
5. Trudność wycinka (**tylko offline**, z wyrocznią): $u = 1 - PQ$ studenta względem wyroczni; pary dopasowane
   (IoU > 0,5) na całym obrazie, para i FP trafiają do wycinka jądra studenta, FN do wycinka jądra wyroczni.
   Zliczenia z wycinków sumują się do zliczeń obrazu.

$$PQ_{\text{wycinek}} = \frac{\sum_{(i,j)\in TP} \mathrm{IoU}(i,j)}{|TP| + \tfrac12|FP| + \tfrac12|FN|}$$

**Oceny:**
- A. korelacja Spearmana wynik wycinka – $u$ wycinka (wszystkie wycinki razem i **wewnątrz obrazu**),
  z 95% przedziałem bootstrap **po obrazach** (wycinki z jednego obrazu nie są niezależne),
- B. czy wynik obrazu złożony z wycinków (najgorszy wycinek, średnia z 2 najgorszych) lepiej szereguje obrazy
  niż średnia po wszystkich jądrach,
- C. podgląd selekcji per wycinek: jaką część różnicy $u$ między średnią puli a idealnym wyborem domyka wybór
  `k` wycinków z najwyższym wynikiem.

Wszystko działa na CPU, jeśli predykcje są w cache; brakujące obrazy student liczy od nowa (na CPU długo).

In [ ]:
SEED = 0  # seed losowego AL, z którego bierzemy studenta
N_LABELED = 4  # runda: student wytrenowany na tylu obrazach
SYMMETRIES = "D4"  # musi się zgadzać z cache notebooka 07 ("D4" albo "D4_SMALL")
ALPHA = 0.2  # próg pokrycia (podział/scalenie/jądra dodatkowe)
RING = 2  # px, pas brzegowy dla U_MI
TILE = 250  # px, bok wycinka (MoNuSeg 1000 px -> 4 x 4)
MIN_NUCLEI = 5  # wycinki z mniejszą liczbą jąder studenta pomijamy w analizach (za duży szum)
MAX_IMAGES = None  # None = wszystkie; na CPU bez cache np. 5
ONLY_CACHED = True  # True: tylko obrazy z gotowymi predykcjami (bez liczenia studenta na CPU)
N_BOOT = 1000  # powtórzenia bootstrap po obrazach
K_TILES = (4, 8, 16, 32)  # liczby wycinków w podglądzie selekcji (32 wycinki = 2 obrazy)

In [ ]:
%matplotlib inline
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from histopathology_pnw.config import Settings
from histopathology_pnw.datasets import monuseg
from histopathology_pnw.device import get_device
from histopathology_pnw.metrics import panoptic_quality
from histopathology_pnw.models.oracle import oracle_masks
from histopathology_pnw.models.student import load_student
from histopathology_pnw.uncertainty import patches, tta

warnings.filterwarnings(
    "ignore", message="An input array is constant"
)  # Spearman przy stałej mierze = NaN
warnings.filterwarnings(
    "ignore", category=RuntimeWarning, message="(Mean of empty slice|All-NaN slice)"
)

SYMS = tta.D4 if SYMMETRIES == "D4" else tta.D4_SMALL
TAG = f"seed{SEED}_n{N_LABELED:02d}_{SYMMETRIES}"
PRED_DIR = Settings().outputs_dir / "tta_uncertainty" / TAG / "predictions"  # cache notebooka 07
OUT = Settings().outputs_dir / "patch_eval" / f"{TAG}_tile{TILE}"
OUT.mkdir(parents=True, exist_ok=True)

seed_dir = Settings().outputs_dir / "al_random" / "full" / f"seed{SEED}"
res = pd.read_csv(seed_dir / "results.csv")
row = res[res["n_labeled"] == N_LABELED].iloc[0]
labeled = json.loads(row["labeled_ids"])
ckpt = Path(row["checkpoint"])
if not ckpt.is_file():
    ckpt = seed_dir / f"round_{N_LABELED:02d}" / "models" / "student"

split = monuseg.load_train_val_split()
by_id = monuseg.samples_by_id("train")
unlabeled = [i for i in split["train"] if i not in labeled]
ids = unlabeled + list(split["val"])
if ONLY_CACHED:
    ids = [i for i in ids if (PRED_DIR / f"{i}.npz").is_file()]
if MAX_IMAGES:
    ids = ids[:MAX_IMAGES]
if not ids:
    raise FileNotFoundError(
        f"brak predykcji TTA w {PRED_DIR}: uruchom najpierw notebook 07 z tymi samymi SEED, "
        "N_LABELED, SYMMETRIES albo ustaw ONLY_CACHED = False (liczenie studenta, długo na CPU)"
    )
part = {i: "pula" for i in unlabeled} | {i: "val" for i in split["val"]}
samples = [by_id[i] for i in ids]
print(f"cache predykcji: {PRED_DIR}")
print(
    f"obrazy: {len(ids)} (pula {sum(part[i] == 'pula' for i in ids)}, val {sum(part[i] == 'val' for i in ids)})"
)

## Wyniki wycinków

Dla każdego obrazu: miary jąder (`tta.nucleus_uncertainty`), wyniki wycinków (`patches.tile_uncertainty`)
i trudność wycinków względem wyroczni (`patches.tile_quality`). Na poziomie obrazu liczymy też wynik
„po staremu” (średnia po wszystkich jądrach), żeby porównać.

`rank_kd` liczymy jak selektor `tta_kd`: rangi procentowe osobno w puli i w walidacji, dla wycinków
tylko spośród tych, które wchodzą do analizy (co najmniej `MIN_NUCLEI` jąder).

In [ ]:
student, tiles, imgs = None, [], []
oracle = dict(zip(ids, oracle_masks(samples), strict=True))
for s in tqdm(samples, desc="obrazy"):
    p = PRED_DIR / f"{s.image_id}.npz"
    if p.is_file():
        pred = tta.TTAPrediction.load(p)
    else:
        if student is None:
            student = load_student(ckpt, get_device())
        pred = tta.predict_tta(student, monuseg.read_image(s.image_path), SYMS)
        PRED_DIR.mkdir(parents=True, exist_ok=True)
        pred.save(p)
    ref, o = pred.masks[0], oracle[s.image_id]
    nuc, st = tta.nucleus_uncertainty(pred, ALPHA, RING)
    grid = patches.TileGrid(*ref.shape[:2], TILE)
    t = patches.tile_uncertainty(nuc, pred, grid, ALPHA).merge(
        patches.tile_quality(ref, o, grid), on="tile"
    )
    tiles.append(t.assign(image_id=s.image_id, part=part[s.image_id]))
    pq = panoptic_quality(o, ref)
    imgs.append(
        {
            "image_id": s.image_id,
            "part": part[s.image_id],
            "u": 1 - pq["pq"],
            "n_nuclei": st["n_nuclei"],
            "K_mean": nuc["K"].mean(),
            "U_MI_mean": nuc["U_MI"].mean(),
            "unstable": (nuc["D"] < 1).mean() + st["E_extra"],
        }
    )

tiles = pd.concat(tiles, ignore_index=True)
imgs = pd.DataFrame(imgs)


def kd_rank(df):
    """Mean percentile rank of K_mean and unstable within each part (as `tta_kd` within the pool)."""
    return df.groupby("part")[["K_mean", "unstable"]].rank(pct=True).mean(axis=1)


use = tiles[(tiles["n_nuclei"] >= MIN_NUCLEI) & tiles["u"].notna()].copy()
use["rank_kd"] = kd_rank(use)
tiles["rank_kd"] = use["rank_kd"]  # NaN for tiles left out of the analysis
imgs["rank_kd"] = kd_rank(imgs)
tiles.to_csv(OUT / "tiles.csv", index=False)
imgs.to_csv(OUT / "images.csv", index=False)
print(f"wycinków: {len(tiles)}, w analizie (>= {MIN_NUCLEI} jąder, u zdefiniowane): {len(use)}")
use[["n_nuclei", "K_mean", "U_MI_mean", "unstable", "u"]].describe(
    percentiles=[0.1, 0.5, 0.9]
).round(3)

## A. Czy wynik wycinka wskazuje trudny wycinek?

- **ρ (wszystkie):** Spearman wynik – $u$ po wszystkich wycinkach razem. Miesza dwa efekty: różnice między
  obrazami i różnice wewnątrz obrazu.
- **ρ (wewnątrz obrazu):** średnia ze Spearmanów liczonych osobno w każdym obrazie. Mówi, czy miara znajduje
  **trudne miejsca w obrębie jednego obrazu**, czyli to, czego średnia po obrazie nie widzi.
- **95% CI:** bootstrap po obrazach (losujemy obrazy ze zwracaniem, bierzemy wszystkie ich wycinki).
- `n_nuclei` to kontrola: czy trudność nie wynika po prostu z liczby jąder.

In [ ]:
SCORES = ["K_mean", "unstable", "rank_kd", "U_MI_mean", "Top_mean", "n_nuclei"]
groups = {i: g for i, g in use.groupby("image_id")}
img_ids = np.array(list(groups))


def rho(d, col):
    return d[col].corr(d["u"], method="spearman")


def rho_within(d, col, min_tiles=4):
    r = [rho(g, col) for _, g in d.groupby("image_id") if len(g) >= min_tiles]
    return float(np.nanmean(r)) if r else np.nan


def boot(fn, col, n=N_BOOT, seed=0):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        pick = rng.choice(img_ids, len(img_ids), replace=True)
        d = pd.concat([groups[i].assign(image_id=f"{i}#{k}") for k, i in enumerate(pick)])
        vals.append(fn(d, col))
    return np.nanpercentile(vals, [2.5, 97.5])


tab = []
for col in SCORES:
    lo, hi = boot(rho, col)
    wlo, whi = boot(rho_within, col)
    tab.append(
        {
            "wynik": col,
            "rho wszystkie": rho(use, col),
            "CI": f"[{lo:.2f}, {hi:.2f}]",
            "rho wewnątrz obrazu": rho_within(use, col),
            "CI wewn.": f"[{wlo:.2f}, {whi:.2f}]",
        }
    )
tab_a = pd.DataFrame(tab)
tab_a.to_csv(OUT / "A_tile_spearman.csv", index=False)
tab_a.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, col in zip(axes, ["K_mean", "unstable", "rank_kd"], strict=True):
    for i, g in use.groupby("image_id"):
        ax.scatter(g[col], g["u"], s=14, alpha=0.7, label=i)
    ax.set_xlabel(col)
    ax.set_ylabel("u = 1 - PQ wycinka")
    ax.set_title(f"{col}: rho = {rho(use, col):.2f}, wewnątrz = {rho_within(use, col):.2f}")
if use["image_id"].nunique() <= 10:
    axes[0].legend(fontsize=7)
plt.tight_layout()
fig.savefig(OUT / "A_scatter.png", dpi=100, bbox_inches="tight")

## B. Wynik obrazu złożony z wycinków

Selekcja dalej wybiera całe obrazy, ale wynik obrazu bierzemy z wycinków:
- `mean jąder`: średnia po wszystkich jądrach obrazu (jak dotąd, wariant `tta_kd`),
- `max wycinka`: najgorszy wycinek,
- `top2 wycinków`: średnia z dwóch najgorszych wycinków.

Wycinki z mniej niż `MIN_NUCLEI` jądrami pomijamy. Przy kilku obrazach to tylko orientacyjne.

In [ ]:
rows = []
for col in ["K_mean", "unstable", "rank_kd"]:
    agg = {"mean jąder": imgs.set_index("image_id")[col]}
    for how, name in (("max", "max wycinka"), ("topk", "top2 wycinków")):
        if col == "rank_kd":
            continue
        agg[name] = tiles.groupby("image_id").apply(
            lambda g, c=col, h=how: patches.image_from_tiles(g, c, h, k=2, min_nuclei=MIN_NUCLEI)
        )
    for name, s in agg.items():
        d = imgs.set_index("image_id").assign(s=s)
        rows.append(
            {
                "miara": col,
                "wynik obrazu": name,
                "rho z u obrazu": d["s"].corr(d["u"], method="spearman"),
                "n obrazów": int(d["s"].notna().sum()),
            }
        )
tab_b = pd.DataFrame(rows)
tab_b.to_csv(OUT / "B_image_from_tiles.csv", index=False)
tab_b.round(3)

## C. Podgląd selekcji per wycinek (offline)

Pula = wycinki nieoznaczonej puli w analizie (walidacji selektor nigdy nie wybiera). Selektor bierze `k` wycinków o najwyższym wyniku; „podgląd” (zagląda
do wyroczni) bierze `k` wycinków o najwyższym $u$; wybór losowy ma w oczekiwaniu średnie $u$ puli.

$$\text{domknięta luka} = \frac{\bar u_{\text{selektor}} - \bar u_{\text{pula}}}{\bar u_{\text{podgląd}} - \bar u_{\text{pula}}}
\qquad (0 = \text{jak losowy},\ 1 = \text{jak podgląd})$$

To nie jest pętla AL (student się nie douczą), tylko sprawdzenie, czy ranking wycinków trafia w trudne wycinki.
Dla porównania ten sam wskaźnik dla obrazów (k = 2 obrazy).

In [ ]:
def gap(df, col, k):
    pool = df["u"].mean()
    sel = df.nlargest(k, col)["u"].mean()
    best = df.nlargest(k, "u")["u"].mean()
    return (sel - pool) / (best - pool) if best > pool else np.nan


pool_t = use[use["part"] == "pula"]
pool_i = imgs[imgs["part"] == "pula"]
print(f"pula: {len(pool_t)} wycinków z {pool_i['image_id'].nunique()} obrazów")
rows = []
for col in ["K_mean", "unstable", "rank_kd", "U_MI_mean"]:
    r = {"wynik": col}
    for k in K_TILES:
        if k < len(pool_t):
            r[f"wycinki k={k}"] = gap(pool_t, col, k)
    if len(pool_i) > 2:
        r["obrazy k=2"] = gap(pool_i.dropna(subset=[col]), col, 2)
    rows.append(r)
tab_c = pd.DataFrame(rows)
tab_c.to_csv(OUT / "C_selection_gap.csv", index=False)
tab_c.round(2)

## Mapa wycinków na obrazie

Ten sam obraz trzy razy: wynik `K_mean`, wynik `unstable` i trudność $u$ z wyroczni. Szare = wycinek pominięty
(mniej niż `MIN_NUCLEI` jąder). Jeśli miary działają lokalnie, ciemne pola powinny się pokrywać.

In [ ]:
IID = (
    use.groupby("image_id")["u"].std().idxmax()
)  # obraz z największym zróżnicowaniem trudności wycinków
img = monuseg.read_image(by_id[IID].image_path)
t = tiles[tiles["image_id"] == IID]
grid = patches.TileGrid(*img.shape[:2], TILE)
fig, axes = plt.subplots(1, 3, figsize=(16, 5.6))
for ax, col in zip(axes, ["K_mean", "unstable", "u"], strict=True):
    m = np.full((grid.n_rows, grid.n_cols), np.nan)
    ok = (t["n_nuclei"] >= MIN_NUCLEI) & t[col].notna()
    m[t.loc[ok, "row"], t.loc[ok, "col"]] = t.loc[ok, col]
    ax.imshow(img)
    im = ax.imshow(
        m,
        cmap="magma_r",
        alpha=0.55,
        extent=(0, grid.n_cols * TILE, grid.n_rows * TILE, 0),
        interpolation="nearest",
    )
    for _, r in t.iterrows():
        txt = f"{r[col]:.2f}" if r["n_nuclei"] >= MIN_NUCLEI and pd.notna(r[col]) else "-"
        ax.text(
            r["x0"] + r["w"] / 2,
            r["y0"] + r["h"] / 2,
            txt,
            ha="center",
            va="center",
            fontsize=9,
            color="white",
            bbox={"facecolor": "black", "alpha": 0.4, "pad": 1, "lw": 0},
        )
    ax.set_xlim(0, img.shape[1])
    ax.set_ylim(img.shape[0], 0)
    ax.set_title(f"{col}" + (" (wyrocznia, offline)" if col == "u" else ""))
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(IID)
plt.tight_layout()
fig.savefig(OUT / "tile_map.png", dpi=100, bbox_inches="tight")

## Wnioski

*(do uzupełnienia po uruchomieniu na pełnych danych)*

- A: czy ρ wewnątrz obrazu jest wyraźnie > 0 (przedział nie obejmuje 0)?
- B: czy „max / top2 wycinków” szereguje obrazy lepiej niż średnia po jądrach?
- C: czy domknięta luka dla wycinków jest wyższa niż dla obrazów? Jeśli tak, to argument za selekcją per wycinek.